# SafeSphere: Behavioural Model Training
Hacksploit 2026 · PS5. This notebook trains the model that tells **stalking / harassment** apart
from **flirting / banter** using behaviour and consent signals.

**Run order:** 1 Setup → 2 Pipeline code → 3 Synthetic data → 4 *Your dataset (optional)* → 5 Train → 6 Evaluate → 7 Try it → 8 Save.
No GPU needed (Runtime → Change runtime type → CPU is fine).

## 1. Setup

In [ ]:
!pip -q install lightgbm pyarrow
import os
os.makedirs('/content/safesphere', exist_ok=True)
%cd /content/safesphere
# Optional: keep everything in Google Drive so it survives a runtime reset
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/safesphere', exist_ok=True)

## 2. Pipeline code
Each cell writes one module. Edit here or in `ml/behavior/` locally and rebuild the notebook.

In [ ]:
%%writefile text_pools.py
"""Hinglish/English text pools per intent, split into disjoint train/test lines.

Lines are combined with fillers and noise at generation time, so a few dozen base lines per
intent produce thousands of distinct messages. Train and test never share a base line.
"""
import random

POOLS = {
    "greeting": [
        "hi", "hello", "hey", "hii", "good morning", "gm", "good night", "gn", "kya haal hai",
        "kaise ho", "hey there", "hello ji", "namaste", "sup", "yo", "hi kaisi ho", "hey wassup",
        "good evening", "morning", "subah subah hi", "hello hello", "hi again", "hey you",
        "kya chal raha hai", "aur batao",
    ],
    "small_talk": [
        "kya kar rahe ho", "khana khaya", "aaj college gaye?", "exam kab hai", "movie dekhi?",
        "weekend plans?", "bahut garmi hai aaj", "kal ka match dekha", "assignment ho gaya?",
        "tired hoon aaj", "what are you doing", "office kaisa tha", "new song suna?",
        "kal milte hain?", "traffic bahut tha", "chai pi li?", "netflix pe kya dekh rahe ho",
        "result aaya?", "class bunk kar di", "mummy ne bulaya", "aaj baarish ho rahi hai",
        "bas aise hi", "kuch nahi yaar", "bored hoon", "lunch kya tha",
    ],
    "flirt": [
        "you look cute today", "tumhari smile bahut pyari hai", "miss you", "thinking about you",
        "you're so pretty", "date pe chalogi?", "aaj bahut acchi lag rahi thi", "can't stop thinking of you",
        "tum special ho", "coffee chalein?", "your dp is so nice", "dil le gayi tum", "you're my favourite",
        "wish you were here", "tumse baat karke accha lagta hai", "you have pretty eyes",
        "kab miloge", "send a selfie na", "you make me smile", "crush ho tum meri",
    ],
    "sexual": [
        "send nudes", "send pics na bas ek baar", "show me your body", "video call pe aao akele",
        "what are you wearing", "come to my room tonight", "i want to see you without clothes",
        "nangi pic bhej", "sexy lag rahi ho", "bed pe kya kar rahi ho", "let's sext",
        "photo bhej na private wali", "tumhari body kamaal hai", "akele milo na raat ko",
        "show me more", "i know you want it too", "ek hot pic bhejo", "send something spicy",
    ],
    "insult": [
        "you are so ugly", "full chapri hai tu", "moti kahin ki", "nobody likes you", "loser",
        "pagal hai kya", "shakal dekhi hai apni", "gawar", "kitni dumb hai tu", "bewakoof",
        "you're worthless", "tu kisi kaam ki nahi", "kutti", "kamini", "go cry somewhere",
        "everyone laughs at you", "kaali kalooti", "tere jaisi ladki", "useless", "cheap lagti hai",
    ],
    "banter_swear": [
        "abe saale", "chal be", "bhai tu pagal hai", "abey chutiye kal kya hua", "saala kamina hai tu",
        "bc kya scene hai", "abe gadhe", "teri toh", "chup kar be", "haha kamine", "ullu ke pathe",
        "tu na sach me", "noob hai tu", "saale treat de", "abe nalle", "bhai kya bakwas hai",
    ],
    "threat": [
        "i will kill you", "dekh lena tujhe", "maar dunga", "chhodunga nahi tujhe", "you will regret this",
        "acid daal dunga", "i will find you", "tera ghar pata hai mujhe", "bahar nikal phir batata hoon",
        "teri life barbaad kar dunga", "you won't be safe", "tujhe sabak sikhaunga",
        "wait and watch kya karta hoon", "tere bhai ko bhi dekh lunga",
    ],
    "sextortion": [
        "10k bhej warna ye photo sabko bhej dunga", "pay or i post your pics", "i have your pics",
        "paise bhej nahi toh viral kar dunga", "your morphed photo is ready", "send money or everyone sees",
        "deepfake bana diya hai tera", "tere ghar walon ko bhej dunga ye photo", "5000 bhej abhi",
        "insta pe daal dunga agar paise nahi diye", "i will leak this", "upi kar warna",
    ],
    "surveillance": [
        "saw you outside college today", "who were you with", "where are you right now",
        "i know where you live", "tum kiske saath thi", "i am outside your house", "send your location",
        "kal metro me dekha tha tujhe", "uss ladke ke saath kya kar rahi thi", "i'm watching you",
        "following you is easy", "tumhara gym time pata hai", "red dress me thi na aaj",
        "aaj late kyu aayi ghar",
    ],
    "clingy": [
        "reply na", "why are you online but not replying", "block kyu kiya", "please reply",
        "ignore mat karo", "answer me", "itna attitude kyu", "ek baar baat kar lo", "hello??",
        "kab tak ignore karogi", "seen karke reply nahi", "i'll keep texting till you reply",
        "pick up the call", "why are you avoiding me", "new account bana liya, ab?", "?", "??",
    ],
    "apology": [
        "sorry, i won't message again", "my bad, sorry", "sorry agar bura laga", "okay, i'll stop",
        "sorry, respect your choice", "theek hai, nahi karunga message", "apologies",
    ],
    "media": ["📷 Photo", "🎥 Video", "🎤 Voice message"],
    # user-side replies
    "warm_reply": [
        "haha you're sweet", "aww thank you", "miss you too", "😊😊", "you too", "haan chalo",
        "pakka milte hain", "hehe stop it 😂", "love you", "same here", "tum bhi", "❤️",
        "accha laga sunke", "yes let's go", "sure, kal?", "😘",
    ],
    "playful_reply": [
        "haha chal be", "😂😂", "abe ja na", "lol", "tu pagal hai", "hahaha", "shut up 😂", "kamine",
        "bakwas mat kar", "lmao", "teri toh 😂", "noob",
    ],
    "neutral_reply": [
        "ok", "hmm", "haan", "theek hai", "fine", "acha", "busy hoon", "baad me baat karte", "okay",
        "ji", "kal batati hoon", "pata nahi", "nahi", "haan khaya",
    ],
    "deflect_reply": [
        "k", "busy", "bhaiya pls", "later", "i have to go", "not now", "hmm ok bye", "sone ja rahi hoon",
        "mummy bula rahi hai", "baad me", "kuch kaam hai",
    ],
    "rejection": [
        "please stop messaging me", "i'm not interested", "mujhe baat nahi karni", "stop it",
        "don't text me again", "leave me alone", "block kar dungi", "please mat karo", "no",
        "i said no", "mujhe pareshan mat karo", "i will report you",
    ],
}

FILLERS_PRE = ["", "", "", "arre ", "yaar ", "suno ", "listen ", "oye ", "bro ", "ji "]
FILLERS_POST = ["", "", "", " yaar", " na", " 😊", " 🙂", " 😂", " pls", " ?", " !!", " 🥺", " 😏"]
LEET = {"a": "@", "i": "1", "e": "3", "o": "0", "s": "$"}


def split_pools(seed=42, test_frac=0.3):
    rng = random.Random(seed)
    train, test = {}, {}
    for k, lines in POOLS.items():
        lines = lines[:]
        rng.shuffle(lines)
        n_test = max(1, int(len(lines) * test_frac))
        test[k], train[k] = lines[:n_test], lines[n_test:]
    return train, test


def _choice(rng, seq):
    return seq[int(rng.integers(len(seq)))]


def noisy(text, rng, p=0.25):
    """Leetspeak, repeated letters, dropped vowels and fillers: the obfuscation real users apply.
    rng is a numpy Generator."""
    if text.startswith(("📷", "🎥", "🎤")):
        return text
    t = text
    if rng.random() < p:
        t = "".join(LEET.get(c, c) if rng.random() < 0.3 else c for c in t)
    if rng.random() < p and len(t) > 3:
        i = int(rng.integers(len(t)))
        t = t[:i] + t[i] * int(rng.integers(2, 5)) + t[i + 1:]
    if rng.random() < p * 0.5:
        t = "".join(c for c in t if c not in "aeiou" or rng.random() < 0.6)
    return (_choice(rng, FILLERS_PRE) + t + _choice(rng, FILLERS_POST)).strip()


def sample_text(pools, intent, rng):
    return noisy(_choice(rng, pools[intent]), rng)

In [ ]:
%%writefile simulate.py
"""Agent-based conversation simulator for the SafeSphere behaviour model.

Each conversation = one other person (possibly several accounts) messaging the user over days.
Scenario parameters are sampled from deliberately OVERLAPPING ranges so classes are not trivially
separable, and ~25% of every safe class are hard negatives (clingy partner, worried parent, ...).

Usage:  python simulate.py --n-train 600 --n-test 150 --out data/synthetic
"""
import argparse
import os
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

from text_pools import sample_text, split_pools

CLASSES = ["safe_mutual", "banter", "normal_frequent", "unreciprocated",
           "stalking", "harassment_after_no", "group_bullying", "sextortion"]
HARMFUL = {"stalking", "harassment_after_no", "group_bullying", "sextortion"}

# intent pool -> message label the text model is expected to output
MSG_LABEL = {
    "greeting": "greeting", "small_talk": "other", "flirt": "flirt", "sexual": "sexual",
    "insult": "insult", "banter_swear": "insult", "threat": "threat", "sextortion": "sextortion",
    "surveillance": "surveillance", "clingy": "clingy", "apology": "apology", "media": "media",
    "warm_reply": "flirt", "playful_reply": "other", "neutral_reply": "other",
    "deflect_reply": "other", "rejection": "rejection",
}
REPLY_LABEL = {"warm_reply": "accept", "playful_reply": "playful", "neutral_reply": "neutral",
               "deflect_reply": "deflect", "rejection": "reject", "banter_swear": "playful"}
APPS = ["wa", "ig", "tg", "sms", "snap"]


def U(rng, a, b):
    return float(rng.uniform(a, b))


def scenario_params(scn, rng):
    """Sample one conversation's parameters. Returns (params, variant_name, is_hard)."""
    p = dict(days=int(rng.integers(5, 31)), rate=U(rng, 3, 20), reply=U(rng, 0.6, 0.9),
             night=U(rng, 0.05, 0.3), burst=0.0, user_init=0.0, latency=U(rng, 5, 60),
             latency_growth=0.0, stages=[(0, {"greeting": 2, "small_talk": 4})],
             reply_mix={"neutral_reply": 3, "warm_reply": 2}, reject_day=None, apology_stop=False,
             post=None, new_accounts=0, block=False, rel="acquaintance", rel_age=U(rng, 30, 900),
             group=0)
    variant, hard = "base", False
    r = rng.random()

    if scn == "safe_mutual":
        p.update(rate=U(rng, 3, 20), reply=U(rng, 0.65, 0.95), night=U(rng, 0.05, 0.35),
                 user_init=U(rng, 0.3, 0.8), rel=rng.choice(["partner", "friend", "acquaintance"]),
                 stages=[(0, {"greeting": 2, "small_talk": 4, "flirt": 3, "sexual": 0.15, "media": 0.1})],
                 reply_mix={"warm_reply": 5, "playful_reply": 2, "neutral_reply": 2, "deflect_reply": 0.3})
        if r < 0.15:
            variant, hard = "clingy_partner", True
            p.update(rel="partner", rate=U(rng, 15, 50), night=U(rng, 0.35, 0.65), reply=U(rng, 0.6, 0.9))
            p["stages"][0][1]["clingy"] = 1.5
        elif r < 0.27:
            variant, hard = "gappy_flirt", True
            p.update(reply=U(rng, 0.3, 0.55), latency=U(rng, 120, 600), user_init=U(rng, 0.2, 0.5))

    elif scn == "banter":
        p.update(rel="friend", rate=U(rng, 3, 25), reply=U(rng, 0.6, 0.95), night=U(rng, 0.05, 0.4),
                 user_init=U(rng, 0.5, 1.0),
                 stages=[(0, {"banter_swear": 3, "small_talk": 3, "greeting": 1, "insult": 0.5})],
                 reply_mix={"playful_reply": 4, "banter_swear": 2, "neutral_reply": 2, "warm_reply": 0.5})
        if r < 0.25:
            variant, hard = "heavy_swearer", True
            p["stages"] = [(0, {"banter_swear": 4, "insult": 2.5, "small_talk": 1})]

    elif scn == "normal_frequent":
        p.update(rel=rng.choice(["family", "friend"]), rate=U(rng, 10, 60), reply=U(rng, 0.5, 0.9),
                 night=U(rng, 0.0, 0.3), user_init=U(rng, 0.3, 0.7),
                 stages=[(0, {"greeting": 2, "small_talk": 6, "clingy": 0.3})],
                 reply_mix={"neutral_reply": 4, "warm_reply": 2, "playful_reply": 1})
        if r < 0.25:
            variant, hard = "worried_parent", True
            p.update(rel="family", night=U(rng, 0.3, 0.55), reply=U(rng, 0.4, 0.8), user_init=U(rng, 0.1, 0.3),
                     stages=[(0, {"small_talk": 3, "clingy": 1.5, "surveillance": 0.5, "greeting": 1})],
                     reply_mix={"neutral_reply": 5, "deflect_reply": 2})

    elif scn == "unreciprocated":
        p.update(rel=rng.choice(["stranger", "acquaintance"]), days=int(rng.integers(3, 21)),
                 rate=U(rng, 1, 10), reply=U(rng, 0.1, 0.4), night=U(rng, 0.05, 0.4), rel_age=U(rng, 0, 120),
                 stages=[(0, {"greeting": 3, "flirt": 2, "small_talk": 2, "clingy": 1})],
                 reply_mix={"neutral_reply": 3, "deflect_reply": 4})
        if r < 0.3:
            variant, hard = "apology_stop", True
            p.update(reject_day=int(rng.integers(1, max(2, p["days"] // 2))), apology_stop=True)

    elif scn == "stalking":
        p.update(rel=rng.choice(["stranger", "acquaintance", "partner"]), rate=U(rng, 8, 60),
                 reply=U(rng, 0.0, 0.2), night=U(rng, 0.3, 0.8), burst=U(rng, 0.1, 0.4),
                 latency=U(rng, 60, 400), latency_growth=U(rng, 0.05, 0.3), rel_age=U(rng, 0, 400),
                 stages=[(0, {"greeting": 3, "clingy": 3, "small_talk": 1, "flirt": 1}),
                         (0.4, {"clingy": 3, "surveillance": 2, "greeting": 1, "flirt": 1, "insult": 0.3})],
                 reply_mix={"neutral_reply": 2, "deflect_reply": 4, "rejection": 1},
                 new_accounts=int(rng.integers(0, 4)), block=rng.random() < 0.6)
        if r < 0.2:
            variant, hard = "polite_stalker", True
            p.update(rate=U(rng, 4, 15), night=U(rng, 0.2, 0.5),
                     stages=[(0, {"greeting": 5, "small_talk": 2, "clingy": 1})])
        elif r < 0.35:
            variant, hard = "fearful_replies", True
            p.update(reply=U(rng, 0.2, 0.4), reply_mix={"neutral_reply": 5, "deflect_reply": 3})

    elif scn == "harassment_after_no":
        p.update(days=int(rng.integers(6, 31)), rate=U(rng, 3, 15), reply=U(rng, 0.5, 0.85),
                 night=U(rng, 0.05, 0.4), user_init=U(rng, 0.1, 0.4),
                 rel=rng.choice(["acquaintance", "partner", "friend"]),
                 stages=[(0, {"greeting": 2, "small_talk": 3, "flirt": 2})],
                 reply_mix={"warm_reply": 2, "neutral_reply": 3, "playful_reply": 1})
        p["reject_day"] = int(p["days"] * U(rng, 0.2, 0.5))
        p["post"] = dict(rate_mult=U(rng, 1.0, 3.0), reply=U(rng, 0.0, 0.12), night=U(rng, 0.2, 0.6),
                         mix={"sexual": 2, "clingy": 3, "insult": 1.5, "flirt": 1.5, "threat": 0.2, "surveillance": 0.5},
                         reply_mix={"deflect_reply": 2, "rejection": 2, "neutral_reply": 1})
        p.update(block=rng.random() < 0.4, new_accounts=int(rng.integers(0, 2)))
        if r < 0.2:
            variant, hard = "polite_after_no", True   # keeps "asking nicely": no sexual/insult words
            p["post"]["mix"] = {"clingy": 3, "flirt": 2, "greeting": 2}

    elif scn == "group_bullying":
        p.update(group=int(rng.integers(3, 7)), days=int(rng.integers(2, 16)), rate=U(rng, 5, 30),
                 reply=U(rng, 0.0, 0.3), night=U(rng, 0.05, 0.4), rel="acquaintance",
                 stages=[(0, {"insult": 5, "banter_swear": 1, "small_talk": 1, "threat": 0.2})],
                 reply_mix={"deflect_reply": 3, "neutral_reply": 2, "rejection": 1})

    elif scn == "sextortion":
        p.update(rel="stranger", rel_age=0.0, days=int(rng.integers(1, 6)), rate=U(rng, 3, 20),
                 reply=U(rng, 0.1, 0.6), night=U(rng, 0.1, 0.5),
                 stages=[(0, {"greeting": 3, "flirt": 3}),
                         (0.3, {"sexual": 3, "flirt": 1, "media": 1}),
                         (0.6, {"sextortion": 4, "threat": 2, "media": 1, "clingy": 1})],
                 reply_mix={"neutral_reply": 2, "deflect_reply": 2, "rejection": 2})
    return p, variant, hard


def _pick(mix, rng):
    keys = list(mix)
    w = np.array([mix[k] for k in keys], dtype=float)
    return keys[rng.choice(len(keys), p=w / w.sum())]


def _stage_mix(stages, progress):
    mix = stages[0][1]
    for start, m in stages:
        if progress >= start:
            mix = m
    return mix


def _msg_time(day0, rng, night):
    hour = rng.uniform(0, 5) if rng.random() < night else rng.uniform(8, 24)
    return day0 + timedelta(hours=float(hour))


def gen_conversation(cid, scn, rng, pools, split, max_sender_msgs=400):
    p, variant, hard = scenario_params(scn, rng)
    start = datetime(2026, 1, 1) + timedelta(days=int(rng.integers(0, 250)))
    apps = list(rng.choice(APPS, size=3, replace=False))
    acc_i, app_i = 0, 0
    if p["group"]:
        accounts = [(f"{cid}_m{i}", apps[0]) for i in range(p["group"])]
    else:
        accounts = [(f"{cid}_a0", apps[0])]
    rows, n_sender = [], 0
    rejected, stopped, blocks_left = False, False, p["new_accounts"]
    block_days = sorted(rng.choice(range(1, max(2, p["days"])), size=min(blocks_left + 1, max(1, p["days"] - 1)),
                                   replace=False).tolist()) if p["block"] else []

    def add(ts, is_user, intent, account, app, reply_label=None, text=None):
        rows.append(dict(conversation_id=cid, person_id=cid, account_id=account, app=app, timestamp=ts,
                         is_user=is_user, text=text if text is not None else sample_text(pools, intent, rng),
                         msg_label=MSG_LABEL.get(intent, "other"), reply_label=reply_label))

    for day in range(p["days"]):
        if stopped or n_sender >= max_sender_msgs:
            break
        day0 = start + timedelta(days=day)
        post = rejected and p["post"] is not None
        rate = p["rate"] * (p["post"]["rate_mult"] if post else 1.0)
        reply_prob = p["post"]["reply"] if post else p["reply"]
        night = p["post"]["night"] if post else p["night"]
        mix = p["post"]["mix"] if post else _stage_mix(p["stages"], day / max(1, p["days"] - 1))
        rmix = p["post"]["reply_mix"] if post else p["reply_mix"]
        latency = p["latency"] * (1 + p["latency_growth"] * day)

        times = [_msg_time(day0, rng, night) for _ in range(rng.poisson(rate))]
        if p["burst"] and rng.random() < p["burst"]:
            b0 = _msg_time(day0, rng, night)
            times += [b0 + timedelta(minutes=float(rng.uniform(0, 30))) for _ in range(rng.integers(10, 16))]
        for t in sorted(times):
            account, app = accounts[rng.integers(len(accounts))] if p["group"] else accounts[-1]
            add(t, False, _pick(mix, rng), account, app)
            n_sender += 1
            if rng.random() < reply_prob:
                kind = _pick(rmix, rng)
                add(t + timedelta(minutes=float(rng.exponential(latency))), True, kind, account, app,
                    REPLY_LABEL.get(kind, "neutral"))
        # user-initiated messages (mutual relationships)
        for _ in range(rng.poisson(rate * p["user_init"] * 0.5)):
            kind = _pick(p["reply_mix"], rng)
            add(_msg_time(day0, rng, 0.1), True, kind, accounts[-1][0], accounts[-1][1], REPLY_LABEL.get(kind, "neutral"))

        if p["reject_day"] is not None and day == p["reject_day"] and not rejected:
            t = day0 + timedelta(hours=float(rng.uniform(18, 23.5)))
            add(t, True, "rejection", accounts[-1][0], accounts[-1][1], "reject")
            rejected = True
            if p["apology_stop"]:
                add(t + timedelta(minutes=float(rng.uniform(1, 120))), False, "apology", *accounts[-1])
                stopped = True
        if block_days and day == block_days[0]:
            block_days.pop(0)
            add(day0 + timedelta(hours=23.9), True, "neutral_reply", accounts[-1][0], accounts[-1][1], "block", text="")
            rows[-1].update(text=None, msg_label="block")
            if blocks_left > 0:
                blocks_left -= 1
                acc_i += 1
                app_i = (app_i + int(rng.random() < 0.5)) % len(apps)
                accounts.append((f"{cid}_a{acc_i}", apps[app_i]))
            else:
                stopped = True

    if scn == "sextortion" and not any(r["msg_label"] == "sextortion" for r in rows):
        last = rows[-1]["timestamp"] if rows else start
        add(last + timedelta(minutes=5), False, "sextortion", *accounts[-1])

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["msg_id"] = np.arange(len(df))
    # hidden notification previews: the text model cannot label these messages
    p_hide = 1.0 if rng.random() < 0.1 else U(rng, 0.1, 0.4)
    hide = (~df["is_user"]) & (rng.random(len(df)) < p_hide)
    df.loc[hide, "text"] = None
    df["user_side_available"] = bool(rng.random() > 0.1)   # live-notification-only conversations
    df["person_label"] = scn
    df["scenario"] = scn
    df["variant"] = variant
    df["hard_case"] = hard
    df["relationship"] = p["rel"]
    df["relationship_age_days"] = float(p["rel_age"])
    df["is_group"] = bool(p["group"])
    df["template_split"] = split
    df["source"] = "synthetic"
    return df


def generate(n_per_class, split, seed, pools):
    rng = np.random.default_rng(seed)
    frames = []
    for scn in CLASSES:
        for i in range(n_per_class):
            df = gen_conversation(f"{split}_{scn}_{i}", scn, rng, pools, split)
            if len(df):
                frames.append(df)
    return pd.concat(frames, ignore_index=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n-train", type=int, default=600)
    ap.add_argument("--n-test", type=int, default=150)
    ap.add_argument("--out", default="data/synthetic")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    train_pools, test_pools = split_pools(seed=42)
    tr = generate(args.n_train, "train", 42, train_pools)
    te = generate(args.n_test, "test", 1337, test_pools)
    tr.to_parquet(f"{args.out}/train.parquet", index=False)
    te.to_parquet(f"{args.out}/test.parquet", index=False)
    for name, df in [("train", tr), ("test", te)]:
        print(f"{name}: {df.conversation_id.nunique()} conversations, {len(df)} messages")
        print(df.groupby("person_label").conversation_id.nunique().to_string())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile features.py
"""Person-level behaviour features. Shared by training and the FastAPI backend.

Input: message rows for ONE person (all linked accounts merged) in the common schema.
Labels come from `obs_label` (what the text model saw); a hidden preview has obs_label='unknown'.
"""
import numpy as np
import pandas as pd

HARMFUL_MSG = {"sexual", "insult", "threat", "sextortion"}
EXPLICIT = {"flirt": 0.5, "sexual": 1.0}
REPLY_KINDS = ["accept", "playful", "neutral", "deflect", "reject", "block"]
RELATIONSHIPS = ["stranger", "acquaintance", "friend", "partner", "family"]
MSG_LABELS = ["greeting", "other", "flirt", "sexual", "insult", "threat", "sextortion",
              "surveillance", "clingy", "apology", "media"]

FEATURES = [
    # volume / timing
    "msgs_per_day", "active_days_ratio", "span_days", "burst_count", "night_ratio", "weekend_ratio",
    # reciprocity
    "reply_ratio", "initiation_ratio", "reply_length_ratio", "unanswered_streak_max",
    "msgs_while_ignored", "user_latency_trend",
    # consent
    "rejected_flag", "msgs_after_reject", "sexual_after_reject", "insult_after_reject",
    "apologised_after_reject", "reply_ratio_after_reject",
    # tone
    "flirt_ratio", "sexual_ratio", "insult_ratio", "clingy_ratio", "threat_count", "sextortion_count",
    "surveillance_count", "media_count", "toxicity_slope", "escalation_gap",
    # user side
    *[f"reply_{k}_share" for k in REPLY_KINDS],
    # evasion
    "accounts_count", "platforms_count", "new_account_after_block", "block_count",
    # context
    *[f"rel_{r}" for r in RELATIONSHIPS], "relationship_age_days", "known_contact", "is_group",
    "text_available_ratio", "user_side_available", "minor_involved",
]

# +1: feature can only increase harm risk; -1: can only decrease it (for the binary head)
MONOTONE = {
    "msgs_after_reject": 1, "sexual_after_reject": 1, "insult_after_reject": 1, "threat_count": 1,
    "sextortion_count": 1, "unanswered_streak_max": 1, "new_account_after_block": 1,
    "surveillance_count": 1, "reply_ratio": -1, "reply_accept_share": -1, "reply_playful_share": -1,
}

# features that need message text (for the metadata-only ablation)
TEXT_FEATURES = ["reply_length_ratio", "sexual_after_reject", "insult_after_reject", "apologised_after_reject",
                 "flirt_ratio", "sexual_ratio", "insult_ratio", "clingy_ratio", "threat_count",
                 "sextortion_count", "surveillance_count", "toxicity_slope", "escalation_gap"]
CONSENT_FEATURES = ["rejected_flag", "msgs_after_reject", "sexual_after_reject", "insult_after_reject",
                    "apologised_after_reject", "reply_ratio_after_reject"]


def add_observed_labels(df, text_labeler=None):
    """obs_label = what the system can see. Hidden previews -> 'unknown'.
    text_labeler(list[str]) -> list[str] replaces ground-truth msg_label with model predictions."""
    df = df.copy()
    if "msg_label" not in df:
        df["msg_label"] = "other"
    if "reply_label" not in df:
        df["reply_label"] = None
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert(None)
    has_text = df["text"].notna() & (df["text"].astype(str).str.len() > 0)
    if text_labeler is not None:
        obs = pd.Series("unknown", index=df.index, dtype=object)
        if has_text.any():
            obs[has_text] = text_labeler(df.loc[has_text, "text"].astype(str).tolist())
    else:
        obs = df["msg_label"].where(has_text, "unknown")
    obs[(df["msg_label"] == "block") | (df["is_user"] & (df["reply_label"] == "block"))] = "block"
    df["obs_label"] = obs
    return df


def apply_label_noise(df, rate, rng):
    """Simulate text-model / reply-classifier errors so the behaviour model is robust to them."""
    df = df.copy()
    known = (df["obs_label"] != "unknown") & (df["obs_label"] != "block") & (~df["is_user"])
    idx = df.index[known & (rng.random(len(df)) < rate)]
    df.loc[idx, "obs_label"] = rng.choice(MSG_LABELS, size=len(idx))
    uidx = df.index[df["is_user"] & df["reply_label"].notna() & (rng.random(len(df)) < rate)]
    df.loc[uidx, "reply_label"] = rng.choice(REPLY_KINDS[:5], size=len(uidx))
    return df


def _slope(y):
    y = np.asarray(y, dtype=float)
    if len(y) < 3 or np.all(y == y[0]):
        return 0.0 if len(y) >= 3 else np.nan
    return float(np.polyfit(np.arange(len(y)), y, 1)[0])


def person_features(g):
    g = g.sort_values("timestamp")
    ts = pd.to_datetime(g["timestamp"])
    user_side = bool(g["user_side_available"].iloc[0]) if "user_side_available" in g else True
    is_block = g["obs_label"] == "block"
    s = g[~g["is_user"]]
    u = g[g["is_user"] & ~is_block] if user_side else g.iloc[0:0]
    f = dict.fromkeys(FEATURES, np.nan)
    ns = len(s)
    if ns == 0:
        return f

    s_ts = pd.to_datetime(s["timestamp"])
    span = max(1.0, (ts.max() - ts.min()).total_seconds() / 86400)
    f["span_days"] = span
    f["msgs_per_day"] = ns / span
    f["active_days_ratio"] = s_ts.dt.normalize().nunique() / np.ceil(span)
    f["night_ratio"] = float((s_ts.dt.hour < 5).mean())
    f["weekend_ratio"] = float((s_ts.dt.dayofweek >= 5).mean())
    secs = (s_ts - s_ts.min()).dt.total_seconds().to_numpy()
    f["burst_count"] = float(np.sum(secs[9:] - secs[:-9] <= 1800)) if ns >= 10 else 0.0

    known = s[s["obs_label"] != "unknown"]
    lab = known["obs_label"]
    nk = max(1, len(known))
    f["text_available_ratio"] = len(known) / ns
    for name, key in [("flirt_ratio", "flirt"), ("sexual_ratio", "sexual"), ("insult_ratio", "insult"),
                      ("clingy_ratio", "clingy")]:
        f[name] = float((lab == key).sum() / nk) if len(known) else np.nan
    for name, key in [("threat_count", "threat"), ("sextortion_count", "sextortion"),
                      ("surveillance_count", "surveillance"), ("media_count", "media")]:
        f[name] = float((s["obs_label"] == key).sum())
    if len(known):
        daily = known.assign(d=pd.to_datetime(known["timestamp"]).dt.normalize(),
                             h=known["obs_label"].isin(HARMFUL_MSG)).groupby("d")["h"].mean()
        f["toxicity_slope"] = _slope(daily.values)

    f["accounts_count"] = float(s["account_id"].nunique())
    f["platforms_count"] = float(s["app"].nunique())
    f["block_count"] = float(is_block.sum())
    if is_block.any():
        t_block = ts[is_block].min()
        first_seen = s.groupby("account_id")["timestamp"].min()
        f["new_account_after_block"] = float((pd.to_datetime(first_seen) > t_block).sum())
    else:
        f["new_account_after_block"] = 0.0

    rel = str(g["relationship"].iloc[0]) if "relationship" in g else "stranger"
    for r in RELATIONSHIPS:
        f[f"rel_{r}"] = float(rel == r)
    f["relationship_age_days"] = float(g["relationship_age_days"].iloc[0]) if "relationship_age_days" in g else np.nan
    f["known_contact"] = float(rel != "stranger")
    f["is_group"] = float(bool(g["is_group"].iloc[0])) if "is_group" in g else 0.0
    f["user_side_available"] = float(user_side)
    f["minor_involved"] = float(bool(g["minor_involved"].any())) if "minor_involved" in g else 0.0
    if not user_side:
        return f   # reciprocity / consent / reply-share stay NaN: the model learns to cope

    # reciprocity on the merged timeline (block events excluded)
    tl = g[~is_block]
    users = tl["is_user"].to_numpy()
    f["reply_ratio"] = len(u) / ns
    runs, cur = [], 0
    for is_u in users:
        if is_u:
            if cur:
                runs.append(cur)
            cur = 0
        else:
            cur += 1
    if cur:
        runs.append(cur)
    f["unanswered_streak_max"] = float(max(runs)) if runs else 0.0
    f["msgs_while_ignored"] = float(sum(r - 1 for r in runs))
    tl_ts = pd.to_datetime(tl["timestamp"]).to_numpy()
    starts = np.r_[True, np.diff(tl_ts).astype("timedelta64[s]").astype(float) > 3 * 3600]
    f["initiation_ratio"] = float((~users[starts]).mean())
    if len(u) and s["text"].notna().any() and u["text"].notna().any():
        f["reply_length_ratio"] = float(u["text"].dropna().str.len().mean() / max(1, s["text"].dropna().str.len().mean()))
    # user reply latency trend (hours, per reply index)
    lat, last_s = [], None
    for is_u, t in zip(users, tl_ts):
        if not is_u:
            last_s = t
        elif last_s is not None:
            lat.append((t - last_s) / np.timedelta64(1, "h"))
            last_s = None
    f["user_latency_trend"] = _slope(lat)

    rl = u["reply_label"].fillna("neutral")
    for k in REPLY_KINDS:
        f[f"reply_{k}_share"] = float((rl == k).mean()) if len(u) else 0.0
    f["reply_block_share"] = float(is_block.sum() / max(1, len(u) + is_block.sum()))

    u_expl = u["obs_label"].map(EXPLICIT).fillna(0).mean() if len(u) else 0.0
    s_expl = known["obs_label"].map(EXPLICIT).fillna(0).mean() if len(known) else np.nan
    f["escalation_gap"] = float(s_expl - u_expl) if len(known) else np.nan

    rej = u[u["reply_label"] == "reject"]
    f["rejected_flag"] = float(len(rej) > 0)
    if len(rej):
        tr = pd.to_datetime(rej["timestamp"]).min()
        after = s[s_ts > tr]
        f["msgs_after_reject"] = float(len(after))
        f["sexual_after_reject"] = float((after["obs_label"] == "sexual").sum())
        f["insult_after_reject"] = float((after["obs_label"] == "insult").sum())
        f["apologised_after_reject"] = float((after["obs_label"] == "apology").any())
        u_after = u[pd.to_datetime(u["timestamp"]) > tr]
        f["reply_ratio_after_reject"] = len(u_after) / max(1, len(after))
    else:
        for k in ["msgs_after_reject", "sexual_after_reject", "insult_after_reject", "apologised_after_reject"]:
            f[k] = 0.0
    return f


def build_feature_table(df, noise=0.0, seed=42, text_labeler=None):
    """Messages -> one row per person. Returns (X, meta) where meta has labels and flags."""
    df = add_observed_labels(df, text_labeler)
    if noise:
        df = apply_label_noise(df, noise, np.random.default_rng(seed))
    rows, meta = [], []
    for pid, g in df.groupby("person_id", sort=False):
        rows.append(person_features(g))
        first = g.iloc[0]
        meta.append(dict(person_id=pid, person_label=first.get("person_label"), variant=first.get("variant", "base"),
                         hard_case=bool(first.get("hard_case", False)), source=first.get("source", "unknown")))
    X = pd.DataFrame(rows, columns=FEATURES)
    return X, pd.DataFrame(meta)

In [ ]:
%%writefile text_model.py
"""Lightweight text models that feed the behaviour model.

- msg model:   sender message text -> msg_label (greeting/flirt/sexual/insult/threat/...)
- reply model: user reply text      -> reply_label (accept/playful/neutral/deflect/reject)

TF-IDF over character n-grams copes with Hinglish spelling variants and leetspeak, trains in
seconds, and is the stand-in until the fine-tuned MuRIL model is ready. Extra labelled messages
from external datasets (HASOC, TRAC, ...) can be added through `extra_msgs`.
"""
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# generic keyword list for the BASELINE only (what a naive filter would use)
BASELINE_KEYWORDS = [
    "nude", "nudes", "sexy", "sex", "body", "naked", "nangi", "bitch", "slut", "whore", "fuck", "bc", "mc",
    "chutiya", "randi", "kutti", "kamini", "ugly", "loser", "kill", "maar", "acid", "rape", "leak", "viral",
    "dekh lena", "chhodunga", "paise bhej", "pay or", "photo sabko",
]


# labels that fire hard rules (critical level) must be predicted confidently
SEVERE = {"threat", "sextortion"}
SEVERE_MIN_PROB = 0.8


def keyword_flag(text):
    t = str(text).lower()
    return any(k in t for k in BASELINE_KEYWORDS)


def _clf():
    return make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
        LogisticRegression(max_iter=2000, class_weight="balanced", C=4.0),
    )


class TextModels:
    def __init__(self):
        self.msg = _clf()
        self.reply = _clf()

    def fit(self, df, extra_msgs=None, max_rows=60000):
        s = df[(~df["is_user"]) & df["text"].notna() & (df["msg_label"] != "block")]
        s = s[["text", "msg_label"]]
        if extra_msgs is not None and len(extra_msgs):
            s = pd.concat([s, extra_msgs[["text", "msg_label"]]], ignore_index=True)
        s = s.drop_duplicates()
        if len(s) > max_rows:
            s = s.sample(max_rows, random_state=42)
        self.msg.fit(s["text"].astype(str), s["msg_label"])
        u = df[df["is_user"] & df["text"].notna() & df["reply_label"].notna() & (df["reply_label"] != "block")]
        u = u[["text", "reply_label"]].drop_duplicates()
        if len(u) > max_rows:
            u = u.sample(max_rows, random_state=42)
        self.reply.fit(u["text"].astype(str), u["reply_label"])
        return self

    def label_messages(self, texts, severe_min_prob=SEVERE_MIN_PROB):
        """Severe labels trigger hard rules downstream, so they need a confident prediction;
        otherwise fall back to the most likely non-severe label."""
        if not len(texts):
            return []
        proba = self.msg.predict_proba(texts)
        classes = np.array(self.msg.classes_)
        severe = np.isin(classes, list(SEVERE))
        best = proba.argmax(1)
        out = classes[best].astype(object)
        weak = severe[best] & (proba.max(1) < severe_min_prob)
        if weak.any():
            fallback = np.where(severe, -1, proba[weak]).argmax(1)
            out[weak] = classes[fallback]
        return list(out)

    def label_replies(self, texts):
        return list(self.reply.predict(texts)) if len(texts) else []

    def relabel(self, df):
        """Replace ground-truth labels with model predictions (realistic end-to-end evaluation)."""
        df = df.copy()
        u = df["is_user"] & df["text"].notna() & (df["text"].astype(str).str.len() > 0)
        if u.any():
            df.loc[u, "reply_label"] = self.label_replies(df.loc[u, "text"].astype(str).tolist())
        return df

    def save(self, path):
        joblib.dump(self, path)

    @staticmethod
    def load(path):
        return joblib.load(path)

In [ ]:
%%writefile train.py
"""Train the SafeSphere behaviour model.

Models: logistic-regression baseline, LightGBM multiclass (which kind of behaviour) and a LightGBM
binary "harmful" head with monotonic constraints + isotonic calibration (how risky).

Usage:  python train.py --train data/synthetic/train.parquet [--extra data/external/conversations.parquet]
"""
import argparse
import json
import os
import time

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from features import FEATURES, MONOTONE, build_feature_table
from simulate import HARMFUL
from text_model import TextModels

THRESHOLDS = {"warn": 0.30, "high": 0.55, "critical": 0.80}
SEED = 42


def logreg():
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                         LogisticRegression(max_iter=3000, class_weight="balanced"))


def tune_lgbm(X, y, n_trials=12):
    """Small random search on macro-F1 (3-fold). Kept small so it runs in minutes on a laptop."""
    rng = np.random.default_rng(SEED)
    skf = StratifiedKFold(3, shuffle=True, random_state=SEED)
    best, best_score = None, -1
    for i in range(n_trials):
        params = dict(num_leaves=int(rng.integers(15, 64)), learning_rate=float(rng.uniform(0.03, 0.1)),
                      min_child_samples=int(rng.integers(10, 51)), colsample_bytree=float(rng.uniform(0.7, 0.9)),
                      n_estimators=int(rng.integers(200, 501)), reg_lambda=float(rng.uniform(0, 2)))
        scores = []
        for tr, va in skf.split(X, y):
            m = lgb.LGBMClassifier(objective="multiclass", class_weight="balanced", random_state=SEED,
                                   verbose=-1, **params)
            m.fit(X.iloc[tr], y.iloc[tr])
            scores.append(f1_score(y.iloc[va], m.predict(X.iloc[va]), average="macro"))
        score = float(np.mean(scores))
        print(f"  trial {i + 1}/{n_trials}: macro-F1={score:.3f}")
        if score > best_score:
            best, best_score = params, score
    return best, best_score


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", default="data/synthetic/train.parquet")
    ap.add_argument("--extra", nargs="*", default=[], help="extra conversations in the common schema (parquet)")
    ap.add_argument("--extra-msgs", default=None, help="labelled messages parquet (text,msg_label) for the text model")
    ap.add_argument("--noise", type=float, default=0.15, help="simulated upstream label-error rate")
    ap.add_argument("--trials", type=int, default=12)
    ap.add_argument("--out", default="models")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    t0 = time.time()

    df = pd.concat([pd.read_parquet(p) for p in [args.train, *args.extra]], ignore_index=True)
    print(f"training data: {df.person_id.nunique()} persons, {len(df)} messages")

    extra_msgs = pd.read_parquet(args.extra_msgs) if args.extra_msgs else None
    tm = TextModels().fit(df, extra_msgs)
    tm.save(f"{args.out}/text_models.pkl")
    print("text models trained")

    X, meta = build_feature_table(df, noise=args.noise, seed=SEED)
    y = meta["person_label"].astype(str)
    yb = y.isin(HARMFUL).astype(int)
    X.assign(person_label=y.values).to_parquet(f"{args.out}/features_train.parquet", index=False)
    print(f"features built: {X.shape}")

    X_fit, X_cal, y_fit, y_cal, yb_fit, yb_cal = train_test_split(
        X, y, yb, test_size=0.2, stratify=y, random_state=SEED)

    print("tuning LightGBM multiclass ...")
    params, cv = tune_lgbm(X_fit, y_fit, args.trials)
    multi = lgb.LGBMClassifier(objective="multiclass", class_weight="balanced", random_state=SEED,
                               verbose=-1, **params).fit(X_fit, y_fit)

    mono = [MONOTONE.get(f, 0) for f in FEATURES]
    binary = lgb.LGBMClassifier(objective="binary", class_weight="balanced", random_state=SEED, verbose=-1,
                                monotone_constraints=mono, monotone_constraints_method="advanced",
                                **params).fit(X_fit, yb_fit)
    iso = IsotonicRegression(out_of_bounds="clip").fit(binary.predict_proba(X_cal)[:, 1], yb_cal)

    lr_multi = logreg().fit(X_fit, y_fit)
    lr_bin = logreg().fit(X_fit, yb_fit)

    bundle = dict(version=time.strftime("%Y%m%d-%H%M"), features=FEATURES, classes=list(multi.classes_),
                  multi=multi, binary=binary, iso=iso, lr_multi=lr_multi, lr_bin=lr_bin,
                  thresholds=THRESHOLDS, params=params, cv_macro_f1=cv, noise=args.noise)
    joblib.dump(bundle, f"{args.out}/behavior_model.pkl")
    val = f1_score(y_cal, multi.predict(X_cal), average="macro")
    json.dump(dict(params=params, cv_macro_f1=cv, holdout_macro_f1=float(val), seconds=round(time.time() - t0)),
              open(f"{args.out}/train_summary.json", "w"), indent=2)
    print(f"done in {time.time() - t0:.0f}s · CV macro-F1 {cv:.3f} · holdout macro-F1 {val:.3f}")
    print(f"saved {args.out}/behavior_model.pkl")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile predict.py
"""Score one person and explain why. This is what the FastAPI backend calls.

    from predict import Scorer
    scorer = Scorer("models")
    result = scorer.score_person(messages_df)
    # {"risk": 0.86, "level": "high", "label": "harassment_after_no", "reasons": [...], ...}

Explanations use LightGBM's exact SHAP contributions (pred_contrib) rendered through fixed
templates. No AI model is involved in the decision or the reasons.
"""
import joblib
import numpy as np
import pandas as pd

from features import FEATURES, add_observed_labels, person_features


def _pct(v):
    return f"{v * 100:.0f}%"


# feature -> (sentence when it pushes risk UP, sentence when it pushes risk DOWN)
TEMPLATES = {
    "msgs_after_reject": (lambda v: f"Sent {v:.0f} messages after you said no", None),
    "sexual_after_reject": (lambda v: f"Sent {v:.0f} sexual messages after you said no", None),
    "insult_after_reject": (lambda v: f"Sent {v:.0f} insults after you said no", None),
    "reply_ratio": (lambda v: f"You replied to only {_pct(v)} of their messages",
                    lambda v: f"You reply to {_pct(min(v, 1))} of their messages"),
    "unanswered_streak_max": (lambda v: f"Sent up to {v:.0f} messages in a row without a reply", None),
    "msgs_while_ignored": (lambda v: f"Kept messaging {v:.0f} times while you didn't respond", None),
    "night_ratio": (lambda v: f"{_pct(v)} of messages were sent between midnight and 5 AM", None),
    "msgs_per_day": (lambda v: f"About {v:.0f} messages per day", None),
    "burst_count": (lambda v: "Sent bursts of 10+ messages within 30 minutes", None),
    "accounts_count": (lambda v: f"Contacted you from {v:.0f} accounts", None),
    "platforms_count": (lambda v: f"Contacted you on {v:.0f} different apps", None),
    "new_account_after_block": (lambda v: f"Created {v:.0f} new account(s) after being blocked", None),
    "surveillance_count": (lambda v: f"{v:.0f} messages tracking where you are or who you're with", None),
    "threat_count": (lambda v: f"{v:.0f} threatening messages", None),
    "sextortion_count": (lambda v: f"{v:.0f} blackmail / sextortion messages", None),
    "sexual_ratio": (lambda v: f"{_pct(v)} of readable messages are sexual", None),
    "insult_ratio": (lambda v: f"{_pct(v)} of readable messages are insults", None),
    "clingy_ratio": (lambda v: f"{_pct(v)} of messages demand replies (\"reply na\", \"why ignoring\")", None),
    "toxicity_slope": (lambda v: "Messages are getting more hostile over time", None),
    "escalation_gap": (lambda v: "They are far more sexual or explicit than you are", None),
    "initiation_ratio": (lambda v: f"They start {_pct(v)} of conversations",
                         lambda v: f"You start {_pct(1 - v)} of conversations"),
    "reply_reject_share": (lambda v: "You have asked them to stop", None),
    "reply_deflect_share": (lambda v: f"{_pct(v)} of your replies avoid the conversation", None),
    "reply_accept_share": (None, lambda v: f"{_pct(v)} of your replies are warm or positive"),
    "reply_playful_share": (None, lambda v: f"{_pct(v)} of your replies are playful banter"),
    "known_contact": (None, lambda v: "This is a known contact"),
    "relationship_age_days": (None, lambda v: f"You have known them for about {v:.0f} days"),
    "apologised_after_reject": (None, lambda v: "They apologised after you said no"),
    "media_count": (lambda v: f"Sent {v:.0f} photo/video/voice files", None),
}


# an "up" reason only makes sense from this value on (e.g. "contacted you from 1 account" is not a reason)
MIN_VALUE = {"accounts_count": 2, "platforms_count": 2, "new_account_after_block": 1, "msgs_after_reject": 1,
             "sexual_after_reject": 1, "insult_after_reject": 1, "surveillance_count": 1, "threat_count": 1,
             "sextortion_count": 1, "media_count": 1, "burst_count": 1, "unanswered_streak_max": 3,
             "night_ratio": 0.2, "sexual_ratio": 0.05, "insult_ratio": 0.05, "clingy_ratio": 0.05,
             "reply_reject_share": 0.01, "reply_deflect_share": 0.1, "toxicity_slope": 0.001,
             "escalation_gap": 0.1, "msgs_while_ignored": 3}


MIN_EVIDENCE_MSGS = 5


def has_evidence(f):
    """Enough behaviour to judge a pattern: 5+ messages from them, or they kept going after a "no".
    The model learned from whole conversations; one or two messages are outside what it was trained on."""
    n_sender = round((f.get("msgs_per_day") or 0) * (f.get("span_days") or 0))
    return n_sender >= MIN_EVIDENCE_MSGS or (f.get("msgs_after_reject") or 0) >= 1


def consistent_label(classes, probs, f):
    """Most likely behaviour type, except "sextortion" needs evidence of it (a demand/threat message or media).
    Short conversations otherwise look like sextortion to the model, because sextortion chats are short."""
    order = np.argsort(-np.asarray(probs))
    evidence = (f.get("sextortion_count") or 0) + (f.get("threat_count") or 0) + (f.get("media_count") or 0)
    readable = (f.get("text_available_ratio") or 0) >= 0.7   # if previews were hidden, absence proves nothing
    for j in order:
        if classes[j] == "sextortion" and evidence == 0 and readable:
            continue
        return classes[j]
    return classes[int(order[0])]


def level_for(risk, f, thresholds):
    level = "info"
    for name in ("warn", "high", "critical"):
        if risk >= thresholds[name]:
            level = name
    # hard rules: never left to the model. One severe message alone does not override clearly mutual,
    # low-risk contact (e.g. friends' banter misread as a threat); two, or one plus some risk, does.
    severe = (f.get("sextortion_count") or 0) + (f.get("threat_count") or 0)
    if (f.get("minor_involved") or 0) >= 1 or severe >= 2 or (severe >= 1 and risk >= thresholds["warn"]):
        return "critical"
    # evidence gate: without enough history the model may only warn
    if level in ("high", "critical") and not has_evidence(f):
        level = "warn"
    return level


class Scorer:
    def __init__(self, model_dir="models", text_models=True):
        self.b = joblib.load(f"{model_dir}/behavior_model.pkl")
        self.tm = None
        if text_models:
            try:
                self.tm = joblib.load(f"{model_dir}/text_models.pkl")
            except FileNotFoundError:
                pass

    def score_features(self, X):
        """X: DataFrame of FEATURES rows -> list of result dicts."""
        b = self.b
        X = X[FEATURES]
        raw = b["binary"].predict_proba(X)[:, 1]
        risk = b["iso"].predict(raw)
        probs = b["multi"].predict_proba(X)
        contrib = b["binary"].booster_.predict(X, pred_contrib=True)[:, :-1]
        out = []
        for i in range(len(X)):
            f = X.iloc[i].to_dict()
            order = np.argsort(-contrib[i])
            up, down = [], []
            for j in order:
                name, c, v = FEATURES[j], contrib[i, j], f[FEATURES[j]]
                t = TEMPLATES.get(name)
                if t is None or v is None or (isinstance(v, float) and np.isnan(v)):
                    continue
                if name in MIN_VALUE and v < MIN_VALUE[name]:
                    continue
                if c > 0.05 and t[0] and len(up) < 5:
                    up.append(t[0](v))
            for j in order[::-1]:
                name, c, v = FEATURES[j], contrib[i, j], f[FEATURES[j]]
                t = TEMPLATES.get(name)
                if t is None or v is None or (isinstance(v, float) and np.isnan(v)):
                    continue
                if c < -0.05 and t[1] and len(down) < 3:
                    down.append(t[1](v))
            out.append(dict(
                risk=round(float(risk[i]), 3),
                level=level_for(float(risk[i]), f, b["thresholds"]),
                early=not has_evidence(f),
                label=consistent_label(b["classes"], probs[i], f),
                label_probs={c: round(float(p), 3) for c, p in zip(b["classes"], probs[i])},
                reasons=up, reasons_down=down,
                facts={k: (None if isinstance(v, float) and np.isnan(v) else round(float(v), 3)) for k, v in f.items()},
                text_available_ratio=f.get("text_available_ratio"),
            ))
        return out

    def score_person(self, msgs):
        """msgs: DataFrame in the common schema for ONE person."""
        if self.tm is not None:
            msgs = self.tm.relabel(msgs)
            df = add_observed_labels(msgs, self.tm.label_messages)
        else:
            df = add_observed_labels(msgs)
        X = pd.DataFrame([person_features(df)], columns=FEATURES)
        return self.score_features(X)[0]

In [ ]:
%%writefile external.py
"""Load outside data into the common schema.

Two kinds of external data are supported:
1. CONVERSATIONS (chat logs with sender + timestamp) -> behaviour-model training/validation.
2. LABELLED MESSAGES (text + label, e.g. HASOC/TRAC/Jigsaw) -> text-model training.

    python external.py inspect path/to/file.csv
    python external.py messages path/to/file.csv --text tweet --label task_2 --map '{"HOF":"insult","NOT":"other"}'
    python external.py conversations path/to/file.csv --conv chat_id --sender from --user-name "Me" \
        --time date --text message --label person_label
    python external.py handwritten data/handwritten
"""
import argparse
import glob
import json
import os

import pandas as pd


def read_any(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".csv", ".tsv"):
        return pd.read_csv(path, sep="\t" if ext == ".tsv" else ",", on_bad_lines="skip")
    if ext in (".json", ".jsonl"):
        try:
            return pd.read_json(path, lines=ext == ".jsonl")
        except ValueError:
            return pd.json_normalize(json.load(open(path, encoding="utf-8")))
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(path)
    raise ValueError(f"unsupported file type: {ext}")


def inspect(path):
    df = read_any(path)
    print(f"{path}: {len(df)} rows, columns: {list(df.columns)}")
    print(df.head(5).to_string())
    for c in df.columns:
        if df[c].dtype == object and df[c].nunique() < 30:
            print(f"\n{c} values:\n{df[c].value_counts().head(15).to_string()}")
    return df


def labelled_messages(path, text_col, label_col, label_map=None):
    df = read_any(path)
    out = pd.DataFrame({"text": df[text_col].astype(str), "msg_label": df[label_col].astype(str)})
    if label_map:
        out["msg_label"] = out["msg_label"].map(label_map)
    return out.dropna()


def conversations(path, conv_col, sender_col, user_name, time_col, text_col, label_col=None,
                  app="wa", relationship="stranger"):
    df = read_any(path)
    out = pd.DataFrame({
        "conversation_id": df[conv_col].astype(str),
        "person_id": df[conv_col].astype(str),
        "account_id": df[conv_col].astype(str) + "_" + df[sender_col].astype(str),
        "app": app,
        "timestamp": pd.to_datetime(df[time_col], errors="coerce"),
        "is_user": df[sender_col].astype(str) == str(user_name),
        "text": df[text_col].astype(str),
        "reply_label": None,
        "person_label": df[label_col].astype(str) if label_col else None,
    })
    out.loc[out.is_user, "account_id"] = "user"
    out = out.dropna(subset=["timestamp"]).sort_values(["conversation_id", "timestamp"])
    out["msg_id"] = out.groupby("conversation_id").cumcount()
    for col, val in [("user_side_available", True), ("relationship", relationship),
                     ("relationship_age_days", float("nan")), ("is_group", False), ("source", "external"),
                     ("variant", "external"), ("hard_case", False)]:
        out[col] = val
    return out


def handwritten(folder):
    """data/handwritten/*.json -> schema. Each file: one conversation or a list of them:
    {"conversation_id", "person_label", "relationship", "relationship_age_days", "is_group", "hard_case",
     "messages": [{"timestamp", "is_user", "text", "app", "account_id", "msg_label"?, "reply_label"?}]}"""
    rows = []
    for fp in sorted(glob.glob(os.path.join(folder, "*.json"))):
        data = json.load(open(fp, encoding="utf-8"))
        for conv in data if isinstance(data, list) else [data]:
            for i, m in enumerate(conv["messages"]):
                rows.append(dict(
                    conversation_id=conv["conversation_id"], person_id=conv["conversation_id"],
                    account_id=m.get("account_id", "user" if m["is_user"] else "a0"), app=m.get("app", "wa"),
                    msg_id=i, timestamp=pd.to_datetime(m["timestamp"]), is_user=bool(m["is_user"]),
                    text=m.get("text"), msg_label=m.get("msg_label"), reply_label=m.get("reply_label"),
                    user_side_available=True, person_label=conv["person_label"],
                    relationship=conv.get("relationship", "stranger"),
                    relationship_age_days=float(conv.get("relationship_age_days", 0)),
                    is_group=bool(conv.get("is_group", False)), hard_case=bool(conv.get("hard_case", False)),
                    variant="handwritten", source="handwritten"))
    return pd.DataFrame(rows)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["inspect", "messages", "conversations", "handwritten"])
    ap.add_argument("path")
    ap.add_argument("--text"); ap.add_argument("--label"); ap.add_argument("--map")
    ap.add_argument("--conv"); ap.add_argument("--sender"); ap.add_argument("--user-name")
    ap.add_argument("--time"); ap.add_argument("--out", default="data/external")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    if a.mode == "inspect":
        inspect(a.path)
    elif a.mode == "messages":
        df = labelled_messages(a.path, a.text, a.label, json.loads(a.map) if a.map else None)
        df.to_parquet(f"{a.out}/messages.parquet", index=False)
        print(df.msg_label.value_counts().to_string())
    elif a.mode == "conversations":
        df = conversations(a.path, a.conv, a.sender, a.user_name, a.time, a.text, a.label)
        df.to_parquet(f"{a.out}/conversations.parquet", index=False)
        print(f"{df.conversation_id.nunique()} conversations, {len(df)} messages")
    else:
        df = handwritten(a.path)
        df.to_parquet(f"{a.out}/handwritten.parquet", index=False)
        print(f"{df.conversation_id.nunique()} conversations")

In [ ]:
%%writefile evaluate.py
"""Evaluate the behaviour model -> reports/metrics.json + charts (feeds the UI Metrics page).

Reports keyword baseline vs logistic regression vs LightGBM: per-class P/R/F1, macro-F1,
binary harmful P/R/F1, detection rate, false-alarm rate on hard negatives, time-to-detect,
and ablations (metadata-only, no consent features).

Usage:  python evaluate.py --test data/synthetic/test.parquet [--handwritten data/external/handwritten.parquet]
"""
import argparse
import json
import os

import joblib
import lightgbm as lgb
import matplotlib
import numpy as np
import pandas as pd
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score)

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

from features import CONSENT_FEATURES, FEATURES, MONOTONE, TEXT_FEATURES, build_feature_table  # noqa: E402
from predict import Scorer  # noqa: E402
from simulate import HARMFUL  # noqa: E402
from text_model import keyword_flag  # noqa: E402

SAFE = ["safe_mutual", "banter", "normal_frequent"]


def binary_block(y_true, flagged):
    return dict(precision=round(float(precision_score(y_true, flagged, zero_division=0)), 3),
                recall=round(float(recall_score(y_true, flagged, zero_division=0)), 3),
                f1=round(float(f1_score(y_true, flagged, zero_division=0)), 3),
                detection_rate=round(float(flagged[y_true == 1].mean()) if (y_true == 1).any() else 0.0, 3))


def far(meta, flagged, hard_only=True):
    m = meta["person_label"].isin(SAFE).to_numpy()
    if hard_only:
        m &= meta["hard_case"].to_numpy()
    return round(float(flagged[m].mean()), 3) if m.any() else None


def keyword_baseline(df):
    s = df[(~df["is_user"]) & df["text"].notna()]
    hit = s.assign(k=s["text"].map(keyword_flag)).groupby("person_id")["k"].any()
    return hit


def evaluate_set(name, df, scorer, use_text_model):
    b = scorer.b
    labeler = scorer.tm.label_messages if (use_text_model and scorer.tm) else None
    if use_text_model and scorer.tm:
        df = scorer.tm.relabel(df)
    X, meta = build_feature_table(df, text_labeler=labeler)
    y = meta["person_label"].astype(str).to_numpy()
    yb = np.isin(y, list(HARMFUL)).astype(int)
    res = scorer.score_features(X)
    flagged = np.array([r["level"] in ("high", "critical") for r in res]).astype(int)
    pred = np.array([r["label"] for r in res])

    kw = keyword_baseline(df).reindex(meta["person_id"]).astype("boolean").fillna(False).to_numpy().astype(int)
    lr_flag = (b["lr_bin"].predict_proba(X)[:, 1] >= 0.5).astype(int)
    lr_pred = b["lr_multi"].predict(X)

    out = dict(
        n_persons=int(len(y)),
        comparison=dict(keyword=binary_block(yb, kw), logreg=binary_block(yb, lr_flag),
                        lightgbm=binary_block(yb, flagged)),
        macro_f1=dict(logreg=round(float(f1_score(y, lr_pred, average="macro")), 3),
                      lightgbm=round(float(f1_score(y, pred, average="macro")), 3)),
        hard_negative_far=dict(keyword=far(meta, kw), logreg=far(meta, lr_flag), lightgbm=far(meta, flagged)),
        all_safe_far=dict(keyword=far(meta, kw, False), logreg=far(meta, lr_flag, False),
                          lightgbm=far(meta, flagged, False)),
        per_class=classification_report(y, pred, output_dict=True, zero_division=0),
        stalking_recall=dict(
            keyword=round(float(kw[y == "stalking"].mean()), 3) if (y == "stalking").any() else None,
            lightgbm=round(float(flagged[y == "stalking"].mean()), 3) if (y == "stalking").any() else None),
    )
    print(f"\n== {name} ({len(y)} persons, text model={'on' if labeler else 'off'}) ==")
    print(pd.DataFrame(out["comparison"]).T.to_string())
    print("macro-F1:", out["macro_f1"], "| hard-negative FAR:", out["hard_negative_far"])
    return out, X, meta, y, pred, res


def time_to_detect(df, scorer, n=60, step=5, seed=0):
    rng = np.random.default_rng(seed)
    pids = df[df["person_label"].isin(HARMFUL)]["person_id"].unique()
    pids = rng.choice(pids, size=min(n, len(pids)), replace=False)
    counts = []
    for pid in pids:
        g = df[df["person_id"] == pid].sort_values("timestamp")
        sender_idx = np.flatnonzero(~g["is_user"].to_numpy())
        hit = None
        for k in range(step, len(sender_idx) + step, step):
            cut = sender_idx[min(k, len(sender_idx)) - 1]
            X, _ = build_feature_table(g.iloc[: cut + 1])
            if scorer.score_features(X)[0]["level"] in ("high", "critical"):
                hit = min(k, len(sender_idx))
                break
        counts.append(hit)
    found = [c for c in counts if c is not None]
    return dict(median_sender_msgs=float(np.median(found)) if found else None,
                detected_share=round(len(found) / max(1, len(counts)), 3), sampled=len(counts))


def early_detection(df, scorer, ks=(3, 5, 10), n=150, seed=0):
    """Score only the first k messages from each person (what the live feed sees early on).
    false_alarm = safe people flagged high+ ; recall = harmful people flagged high+."""
    rng = np.random.default_rng(seed)
    df = scorer.tm.relabel(df) if scorer.tm else df
    labeler = scorer.tm.label_messages if scorer.tm else None
    out = {}
    for group, labels in [("safe", SAFE), ("harmful", sorted(HARMFUL))]:
        pids = df[df["person_label"].isin(labels)]["person_id"].unique()
        pids = rng.choice(pids, size=min(n, len(pids)), replace=False)
        for k in ks:
            prefixes = []
            for pid in pids:
                g = df[df["person_id"] == pid].sort_values("timestamp")
                s_idx = np.flatnonzero(~g["is_user"].to_numpy())
                if len(s_idx) >= k:
                    prefixes.append(g.iloc[: s_idx[k - 1] + 1])
            if not prefixes:
                continue
            X, _ = build_feature_table(pd.concat(prefixes), text_labeler=labeler)
            res = scorer.score_features(X)
            hi = scorer.b["thresholds"]["high"]
            flagged = np.mean([r["level"] in ("high", "critical") for r in res])
            no_gate = np.mean([r["level"] == "critical" or r["risk"] >= hi for r in res])
            key = "false_alarm" if group == "safe" else "recall"
            row = out.setdefault(f"after_{k}_msgs", {})
            row[key] = round(float(flagged), 3)
            row[key + "_without_gate"] = round(float(no_gate), 3)
    print("\nearly detection:", out)
    return out


def ablation(train_feats, X_test, meta_test, b):
    y_tr = train_feats["person_label"].isin(HARMFUL).astype(int)
    yb = meta_test["person_label"].isin(HARMFUL).astype(int).to_numpy()
    out = {}
    for name, drop in [("full", []), ("metadata_only", TEXT_FEATURES), ("no_consent", CONSENT_FEATURES)]:
        Xtr, Xte = train_feats[FEATURES].copy(), X_test[FEATURES].copy()
        if drop:
            Xtr[drop], Xte[drop] = np.nan, np.nan
        m = lgb.LGBMClassifier(objective="binary", class_weight="balanced", random_state=42, verbose=-1,
                               monotone_constraints=[MONOTONE.get(f, 0) for f in FEATURES], **b["params"])
        m.fit(Xtr, y_tr)
        flag = (m.predict_proba(Xte)[:, 1] >= 0.5).astype(int)
        out[name] = dict(**binary_block(yb, flag), hard_negative_far=far(meta_test, flag))
    print("\nablation:\n" + pd.DataFrame(out).T.to_string())
    return out


def charts(out_dir, synth, y, pred, classes, res, meta):
    os.makedirs(out_dir, exist_ok=True)
    cmp = synth["comparison"]
    metrics = ["precision", "recall", "f1", "detection_rate"]
    x = np.arange(len(metrics))
    plt.figure(figsize=(7, 4))
    for i, (k, c) in enumerate([("keyword", "#9aa1b5"), ("logreg", "#c9b8ff"), ("lightgbm", "#6d4aff")]):
        plt.bar(x + (i - 1) * 0.27, [cmp[k][m] for m in metrics], 0.27, label=k, color=c)
    plt.xticks(x, metrics); plt.ylim(0, 1.05); plt.legend(); plt.title("Harmful-behaviour detection")
    plt.tight_layout(); plt.savefig(f"{out_dir}/comparison.png", dpi=150); plt.close()

    cm = confusion_matrix(y, pred, labels=classes, normalize="true")
    plt.figure(figsize=(7, 6)); plt.imshow(cm, cmap="Purples", vmin=0, vmax=1)
    plt.xticks(range(len(classes)), classes, rotation=45, ha="right"); plt.yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            plt.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=8,
                     color="white" if cm[i, j] > 0.5 else "black")
    plt.title("Confusion matrix (row-normalised)"); plt.tight_layout()
    plt.savefig(f"{out_dir}/confusion_matrix.png", dpi=150); plt.close()

    yb = meta["person_label"].isin(HARMFUL).astype(int)
    p, r, _ = precision_recall_curve(yb, [x["risk"] for x in res])
    plt.figure(figsize=(5, 4)); plt.plot(r, p, color="#6d4aff"); plt.xlabel("recall"); plt.ylabel("precision")
    plt.title("PR curve: harmful head"); plt.tight_layout(); plt.savefig(f"{out_dir}/pr_curve.png", dpi=150); plt.close()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", default="models")
    ap.add_argument("--test", default="data/synthetic/test.parquet")
    ap.add_argument("--handwritten", default="data/external/handwritten.parquet")
    ap.add_argument("--external", default="data/external/conversations.parquet")
    ap.add_argument("--cross", default="data/synthetic/test.parquet",
                    help="test set from a DIFFERENT generator (robustness check)")
    ap.add_argument("--out", default="reports")
    ap.add_argument("--skip-ttd", action="store_true")
    args = ap.parse_args()
    scorer = Scorer(args.models)
    test = pd.read_parquet(args.test)

    report = dict(model_version=scorer.b["version"], synthetic_disclosure=True)
    synth, X, meta, y, pred, res = evaluate_set("synthetic test (oracle labels)", test, scorer, False)
    report["synthetic"] = synth
    report["synthetic_text_model"] = evaluate_set("synthetic test (text-model labels)", test, scorer, True)[0]
    for key, path in [("handwritten", args.handwritten), ("external", args.external),
                      ("cross_generator", args.cross)]:
        if os.path.exists(path):
            report[key] = evaluate_set(key, pd.read_parquet(path), scorer, True)[0]
    report["early_detection"] = early_detection(test, scorer)
    if not args.skip_ttd:
        report["time_to_detect"] = time_to_detect(test, scorer)
        print("\ntime to detect:", report["time_to_detect"])
    tf_path = f"{args.models}/features_train.parquet"
    if os.path.exists(tf_path):
        report["ablation"] = ablation(pd.read_parquet(tf_path), X, meta, scorer.b)
    hw = report.get("handwritten")
    if hw and synth["macro_f1"]["lightgbm"] > 0.97 and hw["macro_f1"]["lightgbm"] < 0.75:
        report["warning"] = "Simulator too easy: synthetic >> handwritten. Widen overlaps / add hard negatives."
        print("\n⚠", report["warning"])

    os.makedirs(args.out, exist_ok=True)
    charts(args.out, synth, y, pred, scorer.b["classes"], res, meta)
    with open(f"{args.out}/metrics.json", "w") as fh:
        json.dump(report, fh, indent=2, default=float)
    print(f"\nsaved {args.out}/metrics.json and charts")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile prepare_provided.py
"""Map the team-provided dataset (safesphere_behavior_dataset.zip) onto the pipeline schema.

The provided parquet already follows the common schema; this adds the columns the pipeline
expects and derives the hard-case flag from its `scenario` column.

    python prepare_provided.py --src data/provided/synthetic --out data/prepared
"""
import argparse
import os

import pandas as pd

# scenario values in the provided data that are deliberately tricky
HARD_NEGATIVES = {"welcome_clingy", "long_gaps", "one_no", "worried_parent", "swearing_friend"}
HARD_POSITIVES = {"polite_stalker", "polite_fear_reply"}


def prepare(df):
    df = df.copy()
    df["variant"] = df["scenario"].fillna("standard")
    df["hard_case"] = df["variant"].isin(HARD_NEGATIVES | HARD_POSITIVES)
    has_user = df.groupby("person_id")["is_user"].transform("any")
    df["user_side_available"] = has_user
    df["is_group"] = False   # not provided; group behaviour shows up through accounts_count
    if "minor_involved" not in df:
        df["minor_involved"] = False
    return df


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", default="data/provided/synthetic")
    ap.add_argument("--out", default="data/prepared")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    for split in ("train", "test"):
        df = prepare(pd.read_parquet(f"{a.src}/{split}.parquet"))
        df.to_parquet(f"{a.out}/{split}.parquet", index=False)
        conv = df.drop_duplicates("person_id")
        print(f"{split}: {len(conv)} persons, {len(df)} messages, hard cases {int(conv.hard_case.sum())}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile inspect_provided.py
"""Print the structure of a provided dataset so it can be mapped onto the pipeline schema."""
import sys

import pandas as pd

path = sys.argv[1] if len(sys.argv) > 1 else "data/provided/synthetic/train.parquet"
df = pd.read_parquet(path)
print(path, df.shape)
print(df.dtypes.to_string())
for c in ["msg_label", "reply_label", "person_label", "scenario", "template_split", "relationship", "app",
          "minor_involved", "source", "is_user"]:
    if c in df:
        print(f"\n--- {c} ---\n{df[c].value_counts(dropna=False).head(25).to_string()}")
print("\ntext missing share:", round(df["text"].isna().mean(), 3))
conv = df.groupby("conversation_id")
print("msgs/conv:", conv.size().describe().round(1).to_dict())
print("accounts/person:", df[~df.is_user].groupby("person_id").account_id.nunique().value_counts().head(8).to_dict())
print("persons/conv:", conv.person_id.nunique().value_counts().to_dict())
print("\nscenario x person_label:\n", pd.crosstab(df.drop_duplicates("conversation_id").scenario,
                                                  df.drop_duplicates("conversation_id").person_label).to_string())
print("\nsample rows:\n", df.head(12).to_string())

In [ ]:
%%writefile diagnose.py
"""Why are safe people flagged? Splits false alarms into 'model score' vs 'hard rule' causes.

    python diagnose.py --test data/synthetic/test.parquet
"""
import argparse

import pandas as pd

from evaluate import SAFE
from features import build_feature_table
from predict import Scorer

ap = argparse.ArgumentParser()
ap.add_argument("--models", default="models")
ap.add_argument("--test", default="data/synthetic/test.parquet")
a = ap.parse_args()
sc = Scorer(a.models)
df = sc.tm.relabel(pd.read_parquet(a.test))
X, meta = build_feature_table(df, text_labeler=sc.tm.label_messages)
res = pd.DataFrame(sc.score_features(X))
res = pd.concat([meta, res[["risk", "level", "label"]], X[["threat_count", "sextortion_count", "sexual_ratio"]]], axis=1)
safe = res[res.person_label.isin(SAFE)]
fa = safe[safe.level.isin(["high", "critical"])]
print(f"safe persons: {len(safe)}, flagged: {len(fa)} ({len(fa) / max(1, len(safe)):.1%})")
print(f"  flagged by model score (risk >= 0.55): {(fa.risk >= 0.55).sum()}")
print(f"  flagged ONLY by hard rule (risk < 0.55): {(fa.risk < 0.55).sum()}")
print("\nthreat/sextortion counts among rule-only false alarms:")
print(fa[fa.risk < 0.55][["threat_count", "sextortion_count"]].describe().round(2).to_string())
print("\nfalse alarms by class/variant:\n", fa.groupby(["person_label", "variant"]).size().to_string())

## 3. Load the dataset
**Main data:** the team dataset `safesphere_behavior_dataset.zip` (4,800 train / 1,200 test conversations).
Drag it into the Files pane (left sidebar → 📁 → upload) or run the upload cell below.

We also generate data with our **own simulator** (a different generator with different templates) and train on
**both**. Trained on one generator alone, the model scored F1 0.997 on that generator's test set but only 0.65
on the other's, because it memorised one generator's quirks. Mixing generators fixes that.

In [ ]:
import os
if not os.path.exists('/content/safesphere_behavior_dataset.zip'):
    from google.colab import files
    up = files.upload()   # pick safesphere_behavior_dataset.zip
    for n in up: os.replace(n, f'/content/{n}')
!mkdir -p data/provided && unzip -oq /content/safesphere_behavior_dataset.zip -d data/provided
!python prepare_provided.py --src data/provided/synthetic --out data/prepared
!python simulate.py --n-train 300 --n-test 150 --out data/synthetic

In [ ]:
!python inspect_provided.py data/prepared/train.parquet | head -n 60

## 4. More data (optional)
For any *additional* dataset: upload it, run **inspect** to see the columns, then use one of the converters:
- **Labelled messages** (HASOC / TRAC / Jigsaw: text + label) → improves the text model.
- **Conversations** (chat logs with sender + timestamp) → extra behaviour training / validation.
- **Hand-written test set** (`data/handwritten/*.json`) → honest headline numbers.

In [ ]:
from google.colab import files
os.makedirs('data/raw', exist_ok=True)
uploaded = files.upload()          # pick your dataset file(s)
for name in uploaded:
    os.replace(name, f'data/raw/{name}')
    print('saved', f'data/raw/{name}')

In [ ]:
import glob
for p in glob.glob('data/raw/*'):
    !python external.py inspect "{p}"

In [ ]:
# Fill in the column names after inspecting, then uncomment ONE of these:

# Labelled messages -> data/external/messages.parquet (used by train.py --extra-msgs)
# !python external.py messages data/raw/FILE.csv --text TEXT_COL --label LABEL_COL --map '{"HOF":"insult","NOT":"other"}'

# Conversations -> data/external/conversations.parquet (used by train.py --extra and evaluate.py)
# !python external.py conversations data/raw/FILE.csv --conv CHAT_ID_COL --sender SENDER_COL --user-name "USER" --time TIME_COL --text TEXT_COL --label LABEL_COL

# Hand-written test conversations (folder of JSON files) -> data/external/handwritten.parquet
# !python external.py handwritten data/handwritten

## 5. Train
Keyword baseline, logistic regression, LightGBM multiclass + binary harmful head (monotonic constraints, isotonic calibration). About 2–6 minutes on CPU.

In [ ]:
extra_files = [p for p in ['data/synthetic/train.parquet', 'data/external/conversations.parquet'] if os.path.exists(p)]
if not os.path.exists('data/prepared/train.parquet'):
    extra_files = extra_files[1:]   # synthetic is already the main training set
extra = ('--extra ' + ' '.join(extra_files)) if extra_files else ''
extra_msgs = '--extra-msgs data/external/messages.parquet' if os.path.exists('data/external/messages.parquet') else ''
train_path = 'data/prepared/train.parquet' if os.path.exists('data/prepared/train.parquet') else 'data/synthetic/train.parquet'
!python train.py --train {train_path} {extra} {extra_msgs} --noise 0.15 --trials 8 --out models

## 6. Evaluate
The numbers to report are the **hand-written** set (if available) and the **hard-negative false-alarm rate**, not synthetic accuracy alone.

In [ ]:
test_path = 'data/prepared/test.parquet' if os.path.exists('data/prepared/test.parquet') else 'data/synthetic/test.parquet'
!python evaluate.py --models models --test {test_path} --cross data/synthetic/test.parquet --out reports

In [ ]:
!python diagnose.py --test data/synthetic/test.parquet   # why are safe people flagged: model vs hard rule

In [ ]:
from IPython.display import Image, display
import json
for png in ['comparison', 'confusion_matrix', 'pr_curve']:
    display(Image(f'reports/{png}.png', width=520))
m = json.load(open('reports/metrics.json'))
print(json.dumps({k: m[k] for k in m if k not in ('synthetic', 'synthetic_text_model')}, indent=1)[:3000])

## 7. Try it: score one conversation

In [ ]:
import pandas as pd
from predict import Scorer
scorer = Scorer('models')
test = pd.read_parquet(test_path)
for scn in ['stalking', 'safe_mutual', 'harassment_after_no', 'banter']:
    pid = test[test.person_label == scn].person_id.iloc[0]
    r = scorer.score_person(test[test.person_id == pid])
    print(f"\n[{scn}] risk={r['risk']} level={r['level']} label={r['label']}")
    for s in r['reasons']: print('  ↑', s)
    for s in r['reasons_down']: print('  ↓', s)

## 8. Save the trained model
Download `safesphere_behavior.zip` and unzip into the backend's `models/` folder.

In [ ]:
from google.colab import files
!zip -qr safesphere_behavior.zip models reports
if USE_DRIVE:
    !cp -r models reports safesphere_behavior.zip /content/drive/MyDrive/safesphere/
files.download('safesphere_behavior.zip')